# Local PySpark against the lakehouse

Run this notebook **locally** (not in Docker) with `docker compose up -d` already running.

Setup: `cd spark && uv sync && uv run jupyter notebook`. Requires Java 17+ and the
one-time DNS entry described in `docs/CORS_issues.md`:

```bash
echo "127.0.0.1 minio.localhost" | sudo tee -a /etc/hosts
```

This logs you in as a specific mock user via Keycloak (Resource Owner Password
Credentials — dev-only, never use this grant type in production) and configures
Spark's Iceberg REST catalog with that user's token, so reads/writes are enforced
by Lakekeeper/OpenFGA exactly as declared in `reconciler/grants.yaml`.

Available mock users (password `test1234` for all of them, including the admin):
- `alice`, `bob` — `data-eng-writers`: read + write on `sales.orders`
- `carol`, `dave` — `sales-analytics-readonly`: read-only on `sales.orders` and the `sales_reporting` namespace
- `erin`, `frank` — `sensitive-translation-readers`: read-only on `content.translated_documents_sensitive`
- `vkieuvongngam` — platform admin, full access everywhere

For a non-interactive version of the same thing: `uv run python query_orders.py [user] [password]`.

In [ ]:
# Shared with query_orders.py so there is one implementation.
from query_orders import get_credentials, get_spark_session

## 1. Pick a mock user

Reads `SPARK_USERNAME`/`SPARK_PASSWORD` from `spark/.env` if present (copy
`.env.example` to set one up — no more retyping a password every run), else
prompts interactively. Re-run this cell any time your token expires (default
lifespan: 1 hour).

In [ ]:
username, password = get_credentials()
print(f"Using {username}.")

## 2. Start Spark with the Iceberg REST catalog

In [ ]:
spark = get_spark_session(username, password, "lakehouse-local")
spark.sql("SHOW NAMESPACES").show()

## 3. Read

Should succeed for every mock user granted at least `select` on the table.

In [ ]:
spark.sql("SELECT * FROM sales.orders").show()

## 4. Write

Should succeed for `alice`/`bob` (and `vkieuvongngam`), and fail with a
permission-denied error for `carol`/`dave` (read-only).

In [ ]:
import pandas as pd

data = pd.DataFrame([[1, "acme-corp", 199.99]], columns=["order_id", "customer", "amount"])
spark.createDataFrame(data).writeTo("sales.orders").append()
spark.sql("SELECT * FROM sales.orders").show()